In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tldextract
!pip install -q -U pyarrow
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import pandas as pd

filepath = "/content/drive/MyDrive/out_features.jsonl"

data = []
with open(filepath, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)
print(f"Total rows loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
print("Label distribution:")
print(df['label'].value_counts())

Total rows loaded: 80000
Columns: ['id', 'label', 'source', 'input', 'measurements', 'features']

Label distribution:
label
phishing    40000
benign      40000
Name: count, dtype: int64


In [ ]:
import tldextract

def extract_url(row):
    """URL might be top-level or nested in 'input' — handle both."""
    if isinstance(row.get('url'), str):
        return row['url']
    if isinstance(row.get('input'), dict):
        return row['input'].get('url', '')
    return ''

def get_registered_domain(url):
    if not url:
        return ''
    ext = tldextract.extract(url)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}"
    return ext.domain or ''

def get_tld(url):
    return tldextract.extract(url).suffix if url else ''

df['url_str'] = df.apply(extract_url, axis=1)
df['registered_domain'] = df['url_str'].apply(get_registered_domain)
df['tld'] = df['url_str'].apply(get_tld)

print("Sample:")
print(df[['url_str', 'registered_domain', 'tld', 'label']].head())
print(f"\nRows with empty domain: {(df['registered_domain']=='').sum()}")

Sample:
                                             url_str registered_domain   tld  \
0  https://app46657.swiftway.in/webserver103/?878...       swiftway.in    in   
1               https://quitshadow.com/mko/index.php    quitshadow.com   com   
2                       http://www.twiz.me/u7r88b6k/           twiz.me    me   
3  https://streijl.eu/images/media/inc/naver.com/...        streijl.eu    eu   
4  https://bafkreiaww32bm77n2jnyv3w6v4ncnd7koyalr...         dweb.link  link   

      label  
0  phishing  
1  phishing  
2  phishing  
3  phishing  
4  phishing  

Rows with empty domain: 0


In [ ]:
before = len(df)
df = df[df['url_str'] != ''].reset_index(drop=True)
df = df.drop_duplicates(subset=['url_str']).reset_index(drop=True)
print(f"After removing empty + exact URL duplicates: {before} → {len(df)}")

# How many rows per registered domain (just to see the shape)
domain_counts = df.groupby('registered_domain').size()
print(f"\nUnique registered domains: {len(domain_counts)}")
print(f"Rows per domain — mean: {domain_counts.mean():.1f}, median: {domain_counts.median():.0f}, max: {domain_counts.max()}")

After removing empty + exact URL duplicates: 80000 → 79996

Unique registered domains: 18431
Rows per domain — mean: 4.3, median: 1, max: 3337


In [ ]:
import numpy as np

def domain_aware_split(df, train_size=8000, val_size=1000, test_size=2000, random_state=42):
    """Split such that every registered domain ends up in exactly one split.
    Greedily fills each split with phishing + benign domains until row targets are hit."""

    domain_info = df.groupby('registered_domain').agg(
        label=('label', lambda x: x.mode()[0]),
        count=('label', 'size')
    ).reset_index()

    phish = domain_info[domain_info['label']=='phishing'].sample(
        frac=1, random_state=random_state).reset_index(drop=True)
    benign = domain_info[domain_info['label']=='benign'].sample(
        frac=1, random_state=random_state).reset_index(drop=True)

    splits = {'train': [], 'val': [], 'test': []}
    targets = {'train': train_size, 'val': val_size, 'test': test_size}
    p_i = b_i = 0

    for split_name, total in targets.items():
        half = total // 2
        added = 0
        while added < half and p_i < len(phish):
            splits[split_name].append(phish.iloc[p_i]['registered_domain'])
            added += phish.iloc[p_i]['count']
            p_i += 1
        added = 0
        while added < half and b_i < len(benign):
            splits[split_name].append(benign.iloc[b_i]['registered_domain'])
            added += benign.iloc[b_i]['count']
            b_i += 1

    train_df = df[df['registered_domain'].isin(splits['train'])].reset_index(drop=True)
    val_df   = df[df['registered_domain'].isin(splits['val'])].reset_index(drop=True)
    test_df  = df[df['registered_domain'].isin(splits['test'])].reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = domain_aware_split(df)

print(f"Train: {len(train_df):>5} rows | {train_df['registered_domain'].nunique()} domains")
print(f"Val:   {len(val_df):>5} rows | {val_df['registered_domain'].nunique()} domains")
print(f"Test:  {len(test_df):>5} rows | {test_df['registered_domain'].nunique()} domains")

Train:  8342 rows | 1879 domains
Val:    1026 rows | 332 domains
Test:   2414 rows | 722 domains


In [ ]:
def domain_breakdown(dfx, name, top_n=15):
    counts = dfx.groupby('registered_domain').size().sort_values(ascending=False)
    label_per_domain = dfx.groupby('registered_domain')['label'].first()

    print(f"\n{'='*60}")
    print(f"{name}: {len(counts)} unique domains, {len(dfx)} total rows")
    print(f"{'='*60}")
    print(f"Top {top_n} domains by row count:")
    print(f"  {'domain':<40} {'rows':>6}  label")
    for dom, n in counts.head(top_n).items():
        lbl = label_per_domain[dom]
        print(f"  {dom:<40} {n:>6}  {lbl}")

    print(f"\nDomain count distribution:")
    print(f"  domains with 1 row:     {(counts == 1).sum()}")
    print(f"  domains with 2-10 rows: {((counts >= 2) & (counts <= 10)).sum()}")
    print(f"  domains with 11+ rows:  {(counts >= 11).sum()}")
    print(f"  largest domain:         {counts.iloc[0]} rows ({counts.index[0]})")

domain_breakdown(train_df, "TRAIN")
domain_breakdown(val_df, "VAL")
domain_breakdown(test_df, "TEST")


TRAIN: 1879 unique domains, 8342 total rows
Top 15 domains by row count:
  domain                                     rows  label
  check-gaiastay.com                          559  phishing
  roomtrustverify.com                         542  phishing
  roblox.com.ge                               370  phishing
  duckdns.org                                 198  phishing
  blogspot.com                                194  phishing
  roblox.com.py                               182  phishing
  gitbook.io                                  117  phishing
  shortlink.st                                 83  phishing
  wikipedia.org                                65  benign
  bet83086.com                                 56  phishing
  b45049.com                                   54  phishing
  gobookroom.com                               52  phishing
  staycheckit.com                              50  phishing
  netfimarketing.com                           48  phishing
  b45082.com                   

In [ ]:
train_d = set(train_df['registered_domain'])
val_d   = set(val_df['registered_domain'])
test_d  = set(test_df['registered_domain'])

print("Domain overlap (must all be 0):")
print(f"  Train ∩ Val:  {len(train_d & val_d)}")
print(f"  Train ∩ Test: {len(train_d & test_d)}")
print(f"  Val ∩ Test:   {len(val_d & test_d)}")

print("\nURL overlap (should also be 0 since domains don't overlap):")
print(f"  Train ∩ Test: {len(set(train_df['url_str']) & set(test_df['url_str']))}")

Domain overlap (must all be 0):
  Train ∩ Val:  0
  Train ∩ Test: 0
  Val ∩ Test:   0

URL overlap (should also be 0 since domains don't overlap):
  Train ∩ Test: 0


In [ ]:
for name, dfx in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    counts = dfx['label'].value_counts()
    pcts = dfx['label'].value_counts(normalize=True) * 100
    print(f"\n{name}:")
    for lbl in counts.index:
        print(f"  {lbl}: {counts[lbl]} ({pcts[lbl]:.1f}%)")


Train:
  phishing: 4323 (51.8%)
  benign: 4019 (48.2%)

Val:
  phishing: 522 (50.9%)
  benign: 504 (49.1%)

Test:
  phishing: 1407 (58.3%)
  benign: 1007 (41.7%)


In [ ]:
print("Unique TLDs per split:")
print(f"  Train: {train_df['tld'].nunique()}")
print(f"  Val:   {val_df['tld'].nunique()}")
print(f"  Test:  {test_df['tld'].nunique()}")

print("\nTop 10 TLDs in Train (with phishing share):")
top = train_df['tld'].value_counts().head(10)
for tld, n in top.items():
    sub = train_df[train_df['tld']==tld]
    phish_pct = (sub['label']=='phishing').mean() * 100
    print(f"  .{tld:<8} {n:>5}  ({phish_pct:.0f}% phishing)")

print("\nTop 10 TLDs in Test:")
top = test_df['tld'].value_counts().head(10)
for tld, n in top.items():
    sub = test_df[test_df['tld']==tld]
    phish_pct = (sub['label']=='phishing').mean() * 100
    print(f"  .{tld:<8} {n:>5}  ({phish_pct:.0f}% phishing)")

# How much TLD overlap is there?
common_tlds = set(train_df['tld']) & set(test_df['tld'])
test_only = set(test_df['tld']) - set(train_df['tld'])
print(f"\nTLDs in both Train and Test: {len(common_tlds)}")
print(f"TLDs in Test but NOT in Train: {len(test_only)} (model has never seen these)")

Unique TLDs per split:
  Train: 210
  Val:   73
  Test:  119

Top 10 TLDs in Train (with phishing share):
  .com       4286  (55% phishing)
  .org        554  (38% phishing)
  .com.ge     370  (100% phishing)
  .net        190  (12% phishing)
  .ru         190  (31% phishing)
  .com.py     182  (100% phishing)
  .co.uk      181  (4% phishing)
  .de         147  (14% phishing)
  .io         137  (88% phishing)
  .fr          88  (5% phishing)

Top 10 TLDs in Test:
  .com        901  (46% phishing)
  .dev        440  (100% phishing)
  .org        112  (21% phishing)
  .net        101  (50% phishing)
  .cc          76  (96% phishing)
  .co.uk       56  (12% phishing)
  .de          43  (40% phishing)
  .app         43  (81% phishing)
  .run         38  (100% phishing)
  .ru          33  (27% phishing)

TLDs in both Train and Test: 83
TLDs in Test but NOT in Train: 36 (model has never seen these)


In [ ]:
train_tlds = set(train_df['tld'])
test_tlds = set(test_df['tld'])
val_tlds = set(val_df['tld'])

shared = sorted(train_tlds & test_tlds)
test_only = sorted(test_tlds - train_tlds)
train_only = sorted(train_tlds - test_tlds)

print(f"TLDs shared by Train and Test ({len(shared)}):")
print(f"  {', '.join('.' + t for t in shared)}")

print(f"\nTLDs in Test but NOT in Train ({len(test_only)}):")
print(f"  These are TLDs the model never sees during training:")
for tld in test_only:
    sub = test_df[test_df['tld']==tld]
    n = len(sub)
    phish_pct = (sub['label']=='phishing').mean() * 100
    print(f"  .{tld:<12} {n:>4} rows  ({phish_pct:.0f}% phishing)")

print(f"\nTLDs in Train but NOT in Test ({len(train_only)}):")
print(f"  (model trains on these but won't be tested on them — less of a concern)")
print(f"  {', '.join('.' + t for t in train_only[:30])}{' ...' if len(train_only) > 30 else ''}")

TLDs shared by Train and Test (83):
  .ac.id, .am, .app, .at, .be, .bg, .biz.id, .ca, .cc, .cfd, .cl, .click, .cn, .co, .co.uk, .co.za, .com, .com.br, .com.cn, .com.es, .com.pk, .com.tr, .com.ua, .cyou, .de, .dev, .digital, .edu, .edu.cn, .es, .eu, .finance, .fr, .gov, .gov.uk, .help, .icu, .in, .info, .io, .ir, .it, .jp, .kz, .lat, .live, .lu, .me, .mk, .mx, .my.id, .net, .nl, .no, .onl, .online, .org, .pl, .pro, .qpon, .ro, .rs, .ru, .run, .sbs, .shop, .si, .site, .skin, .su, .tech, .to, .top, .ua, .uk, .us, .video, .vip, .web.id, .win, .world, .xn--p1ai, .xyz

TLDs in Test but NOT in Train (36):
  These are TLDs the model never sees during training:
  .africa          1 rows  (100% phishing)
  .art             1 rows  (0% phishing)
  .az              5 rows  (0% phishing)
  .berlin         11 rows  (0% phishing)
  .bond            1 rows  (100% phishing)
  .cm              2 rows  (100% phishing)
  .co.tz           5 rows  (0% phishing)
  .co.zw           1 rows  (100% phishing)
  .

In [ ]:
from collections import Counter

def feat_count_stats(dfx, name):
    if 'features' not in dfx.columns:
        return
    counts = dfx['features'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    print(f"{name}: features per row — mean {counts.mean():.1f}, median {counts.median():.0f}, max {counts.max()}")

for name, dfx in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    feat_count_stats(dfx, name)

def top_signals(dfx, n=10):
    c = Counter()
    for feats in dfx['features']:
        if isinstance(feats, list):
            for f in feats:
                if isinstance(f, dict) and 'id' in f:
                    c[f['id']] += 1
    return c.most_common(n)

print("\nTop 10 signals in Train:")
for sig, n in top_signals(train_df):
    print(f"  {sig}: {n}")

print("\nTop 10 signals in Test:")
for sig, n in top_signals(test_df):
    print(f"  {sig}: {n}")

Train: features per row — mean 4.6, median 5, max 14
Val: features per row — mean 4.7, median 5, max 12
Test: features per row — mean 4.5, median 5, max 12

Top 10 signals in Train:
  link.null_or_void_anchors: 5165
  navigation.functional_internal_links: 4895
  support.contact_domain_match: 4862
  credential.credential_terms_near_form: 3350
  page.generic_login_without_brand_claim: 2870
  iframe.hidden_iframe: 2616
  form.action_same_org_domain: 2592
  form.hidden_inputs: 1473
  credential.password_input_present: 1398
  contact.identity_domain_mismatch: 1234

Top 10 signals in Test:
  link.null_or_void_anchors: 1226
  navigation.functional_internal_links: 1134
  credential.credential_terms_near_form: 938
  support.contact_domain_match: 902
  credential.password_input_present: 854
  page.generic_login_without_brand_claim: 843
  form.action_same_org_domain: 648
  login.missing_recovery_or_help_flow: 644
  iframe.hidden_iframe: 594
  form.hidden_inputs: 441


In [ ]:
for name, dfx in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    lens = dfx['url_str'].str.len()
    https = dfx['url_str'].str.startswith('https').mean() * 100
    print(f"{name}: URL len mean={lens.mean():.0f}, median={lens.median():.0f}, max={lens.max()} | HTTPS={https:.1f}%")

Train: URL len mean=49, median=37, max=956 | HTTPS=86.9%
Val: URL len mean=50, median=42, max=440 | HTTPS=84.0%
Test: URL len mean=55, median=47, max=803 | HTTPS=74.5%


In [ ]:
SYSTEM_PROMPT = """You are a cybersecurity analyst specialized in detecting phishing websites. You will receive structured information about a webpage and must determine whether it is phishing or benign.

You will receive:
- The URL and final URL (after redirects)
- Page title and visible text excerpt
- Measurements (URL length, form counts, password inputs, etc.)
- Detected signals with severity (low/medium/high), direction (suspicious/safe/uncertain), and a human-readable statement

Weigh high-severity suspicious signals more than low-severity ones. Brand-domain mismatches, credential forms on unfamiliar domains, hidden iframes, and missing recovery flows are strong phishing indicators. Legitimate sites typically have matching brand/domain, proper navigation, and trusted hosting.

Respond in EXACTLY this format:
Classification: <phishing or benign>
Confidence: <low, medium, or high>
Reasoning: <2-4 sentences explaining your decision>"""


def format_input(row):
    """Turn a dataframe row into compact text for the model."""
    inp = row['input'] if isinstance(row.get('input'), dict) else {}
    meas = row['measurements'] if isinstance(row.get('measurements'), dict) else {}
    feats = row['features'] if isinstance(row.get('features'), list) else []

    text = f"URL: {inp.get('url', row.get('url_str', ''))}\n"
    text += f"Final URL: {inp.get('final_url', '')}\n"
    text += f"Title: {inp.get('title', '')}\n"

    visible = inp.get('visible_text', '') or ''
    if len(visible) > 400:
        visible = visible[:400] + "..."
    text += f"Visible text: {visible}\n\n"

    text += "Measurements:\n"
    for k, v in meas.items():
        text += f"  - {k}: {v}\n"

    text += "\nDetected signals:\n"
    if not feats:
        text += "  (none)\n"
    else:
        for f in feats:
            text += (
                f"  - [{f.get('severity','?')}/{f.get('direction','?')}] "
                f"{f.get('id','?')}: {f.get('statement','')}\n"
            )
    return text


def build_assistant_response(row):
    """Build the target response from signals — this is what the model learns to produce."""
    feats = row['features'] if isinstance(row.get('features'), list) else []
    label = row['label']

    if label == 'phishing':
        relevant = [f for f in feats if f.get('direction') == 'suspicious']
    else:
        relevant = [f for f in feats if f.get('direction') == 'safe']

    # Confidence based on number + severity of supporting signals
    high_sev = sum(1 for f in relevant if f.get('severity') == 'high')
    if len(relevant) >= 4 and high_sev >= 1:
        confidence = 'high'
    elif len(relevant) >= 2:
        confidence = 'medium'
    else:
        confidence = 'low'

    # Reasoning from top supporting signals
    top = relevant[:3]
    if top:
        reasoning = '. '.join(f.get('statement', '').rstrip('.') for f in top) + '.'
    else:
        reasoning = (
            "Few clear signals were detected, but on balance the page appears "
            f"{'suspicious' if label=='phishing' else 'legitimate'}."
        )

    return f"Classification: {label}\nConfidence: {confidence}\nReasoning: {reasoning}"


def build_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_input(row)},
        {"role": "assistant", "content": build_assistant_response(row)},
    ]

In [ ]:
sample = train_df.iloc[0]
msgs = build_messages(sample)
print("=" * 60)
print(f"LABEL: {sample['label']}")
print("=" * 60)
print("\n--- USER MESSAGE ---")
print(msgs[1]['content'][:1500])
print("\n--- ASSISTANT RESPONSE (target) ---")
print(msgs[2]['content'])

LABEL: phishing

--- USER MESSAGE ---
URL: https://streijl.eu/images/media/inc/naver.com/index.html
Final URL: https://streijl.eu/images/media/inc/naver.com/index.html
Title: Naver Sign in
Visible text: Naver Sign in Orijinal metin Bu çeviriyi değerlendirin Geri bildiriminiz, Google Çeviri'yi iyileştirmek için kullanılacaktır Dili Seçin Türkçe Abhazca Açece Açoli dili Afar dili Afrikaanca Almanca Alur dili Arapça Arnavutça Assamca Avarca Awadhi dili Aymaraca Azerbaycan dili Bali dili Bambara dili Baoulé dili Baskça Başkurtça Batak Karo dili Batak Simalungun dili Batak Toba dili Batavi dili Belaru...

Measurements:
  - url_length: 56
  - final_url_length: 56
  - final_hostname: streijl.eu
  - hostname_label_count: 2
  - redirect_count: 0
  - form_count: 2
  - credential_form_count: 1
  - password_input_count: 1
  - hidden_input_count: 0
  - form_action_relationship_counts: {}
  - anchor_count: 12
  - external_anchor_count: 7
  - external_anchor_ratio: 0.5833
  - null_anchor_count: 0
  -

In [ ]:
from huggingface_hub import login
login(token="HUGGINGFACE_TOKEN_REMOVED")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


In [ ]:
from datasets import Dataset

def make_dataset(df):
    rows = [{"messages": build_messages(r)} for _, r in df.iterrows()]
    return Dataset.from_list(rows)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)

print(f"Train dataset: {len(train_ds)}")
print(f"Val dataset:   {len(val_ds)}")

Train dataset: 8342
Val dataset:   1026


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

OUTPUT_DIR = "/content/drive/MyDrive/llama3b-phishing-earlystop"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Epoch budget — early stopping will end it sooner if val loss plateaus
    num_train_epochs=10,

    # Batching
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,            # effective batch size = 8

    # Optimization
    learning_rate=2e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # Eval / save / early stopping coordination
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,                       # keep only best + most recent
    load_best_model_at_end=True,              # roll back to best val loss checkpoint
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=20,
    report_to="none",

    # Precision + sequence
    bf16=True,
    max_length=2048,
    packing=False,                            # off so eval loss is per-example, clean
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("✅ Trainer configured. Patience=3, max_epochs=10.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/8342 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1026 [00:00<?, ? examples/s]

✅ Trainer configured. Patience=3, max_epochs=10.


In [ ]:
trainer.train()

print("\n" + "=" * 60)
print("Training complete.")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best eval loss:  {trainer.state.best_metric:.4f}")

# Show the per-epoch eval losses so you can see exactly where it stopped
print("\nEpoch-by-epoch eval loss:")
for log in trainer.state.log_history:
    if 'eval_loss' in log:
        print(f"  Epoch {log['epoch']:.1f}: eval_loss = {log['eval_loss']:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Epoch,Training Loss,Validation Loss
1,0.574309,0.594392
2,0.518588,0.575485
3,0.483460,0.568819
4,0.539454,0.566514
5,0.470061,0.564677
6,0.518712,0.564385
7,0.462749,0.564080
8,0.466682,0.564272
9,0.514415,0.564211
10,0.511433,0.564211



Training complete.
Best checkpoint: /content/drive/MyDrive/llama3b-phishing-earlystop/checkpoint-7301
Best eval loss:  0.5641

Epoch-by-epoch eval loss:
  Epoch 1.0: eval_loss = 0.5944
  Epoch 2.0: eval_loss = 0.5755
  Epoch 3.0: eval_loss = 0.5688
  Epoch 4.0: eval_loss = 0.5665
  Epoch 5.0: eval_loss = 0.5647
  Epoch 6.0: eval_loss = 0.5644
  Epoch 7.0: eval_loss = 0.5641
  Epoch 8.0: eval_loss = 0.5643
  Epoch 9.0: eval_loss = 0.5642
  Epoch 10.0: eval_loss = 0.5642


In [ ]:
ADAPTER_DIR = "/content/drive/MyDrive/llama3b-phishing-best-adapter"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Adapter saved to {ADAPTER_DIR}")

✅ Adapter saved to /content/drive/MyDrive/llama3b-phishing-best-adapter


In [ ]:
import torch, gc

# Drop everything from the previous run
del model, trainer, tokenizer
try:
    del train_ds, val_ds
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

GPU memory allocated: 3.09 GB
GPU memory reserved:  3.92 GB


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME_8B = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_8B)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_8B,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


In [ ]:
from datasets import Dataset

def make_dataset(df):
    rows = [{"messages": build_messages(r)} for _, r in df.iterrows()]
    return Dataset.from_list(rows)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)

print(f"Train dataset: {len(train_ds)}")
print(f"Val dataset:   {len(val_ds)}")

Train dataset: 8342
Val dataset:   1026


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

OUTPUT_DIR_8B = "/content/drive/MyDrive/llama8b-phishing-earlystop"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR_8B,

    num_train_epochs=10,

    # Smaller per-device batch + more accumulation to fit 8B in memory
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,            # effective batch size = 8 (same as 3B)

    # Slightly lower LR — bigger models prefer it
    learning_rate=1e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=20,
    report_to="none",

    bf16=True,
    gradient_checkpointing=True,              # saves memory, slight speed cost
    max_length=2048,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("✅ 8B trainer configured. Patience=3, max_epochs=10.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/8342 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1026 [00:00<?, ? examples/s]

✅ 8B trainer configured. Patience=3, max_epochs=10.


In [30]:
trainer.train()

print("\n" + "=" * 60)
print("Training complete.")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best eval loss:  {trainer.state.best_metric:.4f}")

print("\nEpoch-by-epoch eval loss:")
for log in trainer.state.log_history:
    if 'eval_loss' in log:
        print(f"  Epoch {log['epoch']:.1f}: eval_loss = {log['eval_loss']:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Epoch,Training Loss,Validation Loss
1,0.534426,0.533811
2,0.484150,0.518932
3,0.454145,0.513919
4,0.495442,0.512156
5,0.437426,0.510216
6,0.477350,0.509796


Epoch,Training Loss,Validation Loss
1,0.534426,0.533811
2,0.484150,0.518932
3,0.454145,0.513919
4,0.495442,0.512156
5,0.437426,0.510216
6,0.477350,0.509796


KeyboardInterrupt: 

In [35]:
ADAPTER_DIR_8B = "/content/drive/MyDrive/llama8b-phishing-best-adapter"

model.save_pretrained(ADAPTER_DIR_8B)
tokenizer.save_pretrained(ADAPTER_DIR_8B)
print(f"✅ 8B adapter saved to {ADAPTER_DIR_8B}")

NameError: name 'model' is not defined

In [31]:
import torch, gc

try:
    del model, trainer, tokenizer, train_ds, val_ds
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory allocated: 11.03 GB


In [32]:
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# How many test rows to evaluate on. Start with 200 for a sanity pass,
# bump to len(test_df) once you're happy.
NUM_TEST_SAMPLES = 200

# Adapter paths from earlier cells
ADAPTER_3B = "/content/drive/MyDrive/llama3b-phishing-best-adapter"
ADAPTER_8B = "/content/drive/MyDrive/llama8b-phishing-best-adapter"

BASE_3B = "meta-llama/Llama-3.2-3B-Instruct"
BASE_8B = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


def load_finetuned(base_name, adapter_path):
    tok = AutoTokenizer.from_pretrained(adapter_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        base_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    mdl = PeftModel.from_pretrained(base, adapter_path)
    mdl.eval()
    return mdl, tok


def ask(model, tokenizer, user_content, max_new_tokens=180):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)


def parse_response(text):
    """Extract classification, confidence, reasoning. Return (label, confidence, reasoning)."""
    label = None
    if re.search(r'classification\s*:\s*phishing', text, re.IGNORECASE):
        label = 'phishing'
    elif re.search(r'classification\s*:\s*benign', text, re.IGNORECASE):
        label = 'benign'
    else:
        # Fallback: first occurrence of phishing/benign anywhere
        m = re.search(r'\b(phishing|benign)\b', text, re.IGNORECASE)
        label = m.group(1).lower() if m else None

    conf_match = re.search(r'confidence\s*:\s*(low|medium|high)', text, re.IGNORECASE)
    confidence = conf_match.group(1).lower() if conf_match else None

    reason_match = re.search(r'reasoning\s*:\s*(.+)', text, re.IGNORECASE | re.DOTALL)
    reasoning = reason_match.group(1).strip() if reason_match else ''

    return label, confidence, reasoning


def evaluate_model(model, tokenizer, test_subset, model_name):
    """Run model on test_subset, return predictions + metadata."""
    results = []
    print(f"\nEvaluating {model_name} on {len(test_subset)} samples...")
    for i, (_, row) in enumerate(test_subset.iterrows()):
        if i % 25 == 0:
            print(f"  {i}/{len(test_subset)}")
        user_msg = format_input(row)
        raw = ask(model, tokenizer, user_msg)
        pred, conf, reasoning = parse_response(raw)
        results.append({
            'url': row['url_str'],
            'true_label': row['label'],
            'pred_label': pred,
            'confidence': conf,
            'reasoning': reasoning,
            'raw_response': raw,
        })
    return results

In [33]:
# Stratified subsample: equal phishing and benign from test_df
n_per_class = NUM_TEST_SAMPLES // 2
test_subset = (
    test_df.groupby('label', group_keys=False)
           .apply(lambda x: x.sample(n=min(n_per_class, len(x)), random_state=42))
           .reset_index(drop=True)
)
print(f"Evaluation set: {len(test_subset)} samples")
print(test_subset['label'].value_counts())

# --- 3B ---
model_3b, tok_3b = load_finetuned(BASE_3B, ADAPTER_3B)
results_3b = evaluate_model(model_3b, tok_3b, test_subset, "Llama-3.2-3B (fine-tuned)")
del model_3b, tok_3b
gc.collect(); torch.cuda.empty_cache()

# --- 8B ---
model_8b, tok_8b = load_finetuned(BASE_8B, ADAPTER_8B)
results_8b = evaluate_model(model_8b, tok_8b, test_subset, "Llama-3.1-8B (fine-tuned)")
del model_8b, tok_8b
gc.collect(); torch.cuda.empty_cache()

print("\n✅ Evaluation done.")

/tmp/ipykernel_10779/3998066426.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(n_per_class, len(x)), random_state=42))


Evaluation set: 200 samples
label
benign      100
phishing    100
Name: count, dtype: int64


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Evaluating Llama-3.2-3B (fine-tuned) on 200 samples...
  0/200


AttributeError: 

In [34]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

def metrics_table(results, model_name):
    df_r = pd.DataFrame(results)
    parsed_ok = df_r['pred_label'].notna()
    parse_failures = (~parsed_ok).sum()

    # Treat parse failures as wrong (predict opposite of true label)
    df_r.loc[~parsed_ok, 'pred_label'] = df_r.loc[~parsed_ok, 'true_label'].apply(
        lambda x: 'benign' if x == 'phishing' else 'phishing'
    )

    y_true = df_r['true_label']
    y_pred = df_r['pred_label']
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, pos_label='phishing', average='binary'
    )

    print(f"\n{'='*60}")
    print(f"{model_name}")
    print(f"{'='*60}")
    print(f"Parse failures: {parse_failures}/{len(df_r)}")
    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}  (of predicted phishing, how many are correct)")
    print(f"Recall:    {rec:.3f}  (of actual phishing, how many we caught)")
    print(f"F1:        {f1:.3f}")
    print()
    print(classification_report(y_true, y_pred, digits=3))
    print("Confusion matrix (rows=true, cols=pred):")
    cm_labels = ['benign', 'phishing']
    cm = confusion_matrix(y_true, y_pred, labels=cm_labels)
    print(f"          {'  '.join(f'{l:>9}' for l in cm_labels)}")
    for i, l in enumerate(cm_labels):
        print(f"  {l:<8} {'  '.join(f'{v:>9}' for v in cm[i])}")
    return {'model': model_name, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
            'parse_fail': parse_failures}

m3 = metrics_table(results_3b, "Llama-3.2-3B (fine-tuned)")
m8 = metrics_table(results_8b, "Llama-3.1-8B (fine-tuned)")

print("\n" + "=" * 60)
print("HEAD-TO-HEAD")
print("=" * 60)
summary = pd.DataFrame([m3, m8])
print(summary.to_string(index=False))

NameError: name 'results_3b' is not defined

In [ ]:
df3 = pd.DataFrame(results_3b).rename(columns={
    'pred_label': 'pred_3b', 'confidence': 'conf_3b', 'reasoning': 'reason_3b'
})[['url', 'true_label', 'pred_3b', 'conf_3b', 'reason_3b']]

df8 = pd.DataFrame(results_8b).rename(columns={
    'pred_label': 'pred_8b', 'confidence': 'conf_8b', 'reasoning': 'reason_8b'
})[['url', 'pred_8b', 'conf_8b', 'reason_8b']]

merged = df3.merge(df8, on='url')
disagreements = merged[merged['pred_3b'] != merged['pred_8b']]

print(f"Disagreements: {len(disagreements)}/{len(merged)} samples")
print(f"  3B right, 8B wrong: {((disagreements['pred_3b']==disagreements['true_label'])).sum()}")
print(f"  8B right, 3B wrong: {((disagreements['pred_8b']==disagreements['true_label'])).sum()}")
print(f"  Both wrong (different ways): {((disagreements['pred_3b']!=disagreements['true_label']) & (disagreements['pred_8b']!=disagreements['true_label'])).sum()}")

# Show 5 disagreement cases
print("\n" + "=" * 80)
print("Sample disagreements:")
print("=" * 80)
for _, row in disagreements.head(5).iterrows():
    print(f"\nURL: {row['url']}")
    print(f"TRUE: {row['true_label']}")
    print(f"  3B → {row['pred_3b']} ({row['conf_3b']}): {row['reason_3b'][:150]}")
    print(f"  8B → {row['pred_8b']} ({row['conf_8b']}): {row['reason_8b'][:150]}")
    print("-" * 80)

In [ ]:
pd.DataFrame(results_3b).to_csv("/content/drive/MyDrive/results_3b.csv", index=False)
pd.DataFrame(results_8b).to_csv("/content/drive/MyDrive/results_8b.csv", index=False)
merged.to_csv("/content/drive/MyDrive/results_merged.csv", index=False)
print("✅ Saved results_3b.csv, results_8b.csv, results_merged.csv to Drive")

In [36]:
from google.colab import runtime
runtime.unassign()